In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
import os
from langgraph.checkpoint.memory import InMemorySaver  # , for saving state into ram memory
from langchain_groq import ChatGroq



In [2]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")



In [3]:
model = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

In [4]:
class JokeState(TypedDict):

    topic:str
    joke:str
    explanation:str
    

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = model.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.',
 'explanation': 'This joke is a play on words, using a common phrase to create a pun. The phrase "feeling a little crusty" is typically used to describe someone who is being irritable or gruff, often due to being tired, stressed, or in a bad mood.\n\nIn this joke, the phrase is applied to a pizza, which has a crust as one of its main components. The word "crusty" has a double meaning here - it refers to the pizza\'s crust, but also to the phrase "feeling a little crusty" which means being irritable.\n\nThe joke relies on this wordplay to create humor. The punchline "it was feeling a little crusty" is a clever and unexpected twist on the usual meaning of the phrase, and the fact that it\'s applied to a pizza (an object that actually has a crust) adds to the comedic effect. The joke requires a quick mental shift to understand the double meaning of "crusty", and the surprise and clev

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, using a common phrase to create a pun. The phrase "feeling a little crusty" is typically used to describe someone who is being irritable or gruff, often due to being tired, stressed, or in a bad mood.\n\nIn this joke, the phrase is applied to a pizza, which has a crust as one of its main components. The word "crusty" has a double meaning here - it refers to the pizza\'s crust, but also to the phrase "feeling a little crusty" which means being irritable.\n\nThe joke relies on this wordplay to create humor. The punchline "it was feeling a little crusty" is a clever and unexpected twist on the usual meaning of the phrase, and the fact that it\'s applied to a pizza (an object that actually has a crust) adds to the comedic effect. The joke requires a quick mental shift to understand the double meaning of "crusty", and th

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, using a common phrase to create a pun. The phrase "feeling a little crusty" is typically used to describe someone who is being irritable or gruff, often due to being tired, stressed, or in a bad mood.\n\nIn this joke, the phrase is applied to a pizza, which has a crust as one of its main components. The word "crusty" has a double meaning here - it refers to the pizza\'s crust, but also to the phrase "feeling a little crusty" which means being irritable.\n\nThe joke relies on this wordplay to create humor. The punchline "it was feeling a little crusty" is a clever and unexpected twist on the usual meaning of the phrase, and the fact that it\'s applied to a pizza (an object that actually has a crust) adds to the comedic effect. The joke requires a quick mental shift to understand the double meaning of "crusty", and t

Time Travel Example

In [11]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f14cc68-e812-6128-8000-72958391a576"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f14cc68-e812-6128-8000-72958391a576'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-05-10T23:18:30.325888+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f14cc68-e811-6048-bfff-f0337d845e67'}}, tasks=(PregelTask(id='2869741f-6d9e-3c75-acb7-3ef43d789892', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.'}),), interrupts=())

In [15]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f14cc6a-effa-667c-8001-9f521579b961"}}) # resume from this checkpoint id 1f14cc6a-effa-667c-8001-9f521579b961 and update

{'topic': 'samosa',
 'joke': 'Why did the samosa go to therapy?\n\nBecause it was feeling a little "crunchy" on the outside and "empty" on the inside.',
 'explanation': 'This joke is a play on words, using the physical characteristics of a samosa to make a humorous comment on emotional well-being. \n\nA samosa is a type of fried or baked pastry that is typically crunchy on the outside and filled with ingredients like spiced potatoes, peas, and onions. In this joke, the samosa\'s crunchy exterior and empty interior are used as metaphors for emotional states.\n\nThe phrase "crunchy on the outside" suggests a tough or hardened exterior, which can be seen as a defense mechanism or a way of hiding one\'s true feelings. On the other hand, "empty on the inside" implies a sense of hollowness or lack of fulfillment, which can be a common feeling in people who are struggling with emotional issues.\n\nThe joke is saying that the samosa went to therapy because it was struggling with these feelings

In [16]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to therapy?\n\nBecause it was feeling a little "crunchy" on the outside and "empty" on the inside.', 'explanation': 'This joke is a play on words, using the physical characteristics of a samosa to make a humorous comment on emotional well-being. \n\nA samosa is a type of fried or baked pastry that is typically crunchy on the outside and filled with ingredients like spiced potatoes, peas, and onions. In this joke, the samosa\'s crunchy exterior and empty interior are used as metaphors for emotional states.\n\nThe phrase "crunchy on the outside" suggests a tough or hardened exterior, which can be seen as a defense mechanism or a way of hiding one\'s true feelings. On the other hand, "empty on the inside" implies a sense of hollowness or lack of fulfillment, which can be a common feeling in people who are struggling with emotional issues.\n\nThe joke is saying that the samosa went to therapy because it was struggling

Updating State

In [17]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f14cc68-e812-6128-8000-72958391a576", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f14cc6c-c051-6806-8001-6c362d3e1bd4'}}